In [16]:
# IN12: Logging, Tracing, Observability and Debugging

In [17]:
# Objectives

# By the end of this notebook you will be able to:
# - Understand the trace/span/event hierarchy for agentic systems
# - Implement structured logging with correlation IDs
# - Attribute latency across pipeline stages
# - Compute per-trace token cost and cumulative spend
# - Integrate Langfuse for production observability (with graceful fallback)
# - Perform root-cause debugging using trace data
# - Build an evaluation scorecard and observability dashboard design document

# Deliverables: evaluation_scorecard.txt, observability_dashboard_design.txt

In [18]:
# User query → Document retrieval → LLM call → Tool/API call → Final response

# 1. Traces, spans and events
# 2. Structured logging
# 3. Latency attribution
# 4. Token and cost monitoring
# 5. Langfuse integration
# 6. Root-cause debugging
# 7. Evaluation scorecard and dashboard

# Final Objective:
# The notebook builds an observability system that answers: “What happened during this request, where did the problem occur, and what evidence do we have to fix it?”

In [19]:
import os, json, time, uuid, logging
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
OPENAI_API_KEY       = os.getenv('OPENAI_API_KEY')
LANGFUSE_PUBLIC_KEY  = os.getenv('LANGFUSE_PUBLIC_KEY', '')
LANGFUSE_SECRET_KEY  = os.getenv('LANGFUSE_SECRET_KEY', '')

client = OpenAI(api_key=OPENAI_API_KEY)
print('Client ready.')
print(f'Langfuse keys present: {bool(LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY)}')

Client ready.
Langfuse keys present: True


## Section 1: Trace / Span / Event -- The Observability Hierarchy

### What is a Trace?

A trace is the complete record of a single end-to-end request through a system.
It captures every step from the moment the user's query arrives to the moment
the final response is delivered.

**The three-level hierarchy:**

| Level | Definition | Analogy |
|---|---|---|
| Trace | One complete request lifecycle | A receipt for the entire transaction |
| Span | One named unit of work within the trace | A line item on the receipt |
| Event | A point-in-time record within a span | A timestamp printed mid-step |

**For the Walmart Retail Assistant, one trace contains these spans:**

```
Trace [trace_id: T-001, session: S-42, user_query: 'milk price']
  Span [retrieval]     - Pinecone query, K=3 chunks returned
  Span [llm_call]      - gpt-4-turbo, 120 input tokens, 45 output tokens
  Span [tool_dispatch] - price_lookup tool called with sku=GV-MILK-1G
  Span [response]      - Final answer assembled and returned
```

**What each span records:**

| Attribute | Description |
|---|---|
| span_id | Unique identifier for this span |
| parent_trace_id | Links span back to its trace |
| name | Human-readable label (e.g., 'retrieval', 'llm_call') |
| start_time | UTC timestamp when the span started |
| end_time | UTC timestamp when the span ended |
| duration_ms | end - start in milliseconds |
| status | success / error / timeout |
| metadata | Arbitrary key-value pairs (tokens, scores, tool names) |

**What to remember:**
- Every trace needs a unique `trace_id`. Use UUID4 -- never a sequential counter.
  Sequential IDs leak information about request volume.
- Always record both start and end times. Duration computed post-hoc from logs
  loses sub-millisecond precision and cannot be aggregated reliably.
- Span names must be consistent. 'llm_call' and 'LLM Call' in the same system
  create split metrics in every dashboard.
- Store `trace_id` in every log line. Without it, correlating events across spans
  in a distributed system becomes infeasible.

In [20]:
# Trace / Span implementation for Walmart Retail Assistant
# Ex:
# Customer asks: “Where is my order?”
# That complete request is one trace, while retrieval, order API call, LLM generation, and response validation are separate spans.

class Span:
    """Represents one unit of work within a trace."""
    def __init__(self, name: str, trace_id: str, metadata: dict = None):
        self.span_id   = str(uuid.uuid4())[:8]
        self.trace_id  = trace_id
        self.name      = name
        self.start_ts  = datetime.now(timezone.utc).isoformat()
        self.start_ns  = time.monotonic_ns()
        self.end_ns    = None
        self.status    = 'in_progress'
        self.metadata  = metadata or {}
        self.events    = []

    def add_event(self, name: str, data: dict = None):
        self.events.append({
            'name': name,
            'ts':   datetime.now(timezone.utc).isoformat(),
            'data': data or {},
        })

    def finish(self, status: str = 'success', metadata: dict = None):
        self.end_ns  = time.monotonic_ns()
        self.status  = status
        if metadata:
            self.metadata.update(metadata)

    @property
    def duration_ms(self) -> float:
        if self.end_ns is None:
            return 0.0
        return round((self.end_ns - self.start_ns) / 1_000_000, 2)

    def to_dict(self) -> dict:
        return {
            'span_id':     self.span_id,
            'trace_id':    self.trace_id,
            'name':        self.name,
            'start_ts':    self.start_ts,
            'duration_ms': self.duration_ms,
            'status':      self.status,
            'metadata':    self.metadata,
            'events':      self.events,
        }


class Trace:
    """Represents one complete request lifecycle."""
    def __init__(self, session_id: str, user_query: str):
        self.trace_id    = str(uuid.uuid4())
        self.session_id  = session_id
        self.user_query  = user_query
        self.start_ts    = datetime.now(timezone.utc).isoformat()
        self.start_ns    = time.monotonic_ns()
        self.spans: list[Span] = []
        self.status      = 'in_progress'

    def new_span(self, name: str, metadata: dict = None) -> Span:
        s = Span(name, self.trace_id, metadata)
        self.spans.append(s)
        return s

    def finish(self, status: str = 'success'):
        self.end_ns = time.monotonic_ns()
        self.status = status

    @property
    def total_duration_ms(self) -> float:
        if not hasattr(self, 'end_ns'):
            return 0.0
        return round((self.end_ns - self.start_ns) / 1_000_000, 2)

    def to_dict(self) -> dict:
        return {
            'trace_id':         self.trace_id,
            'session_id':       self.session_id,
            'user_query':       self.user_query,
            'start_ts':         self.start_ts,
            'total_duration_ms':self.total_duration_ms,
            'status':           self.status,
            'span_count':       len(self.spans),
            'spans':            [s.to_dict() for s in self.spans],
        }

print('Trace and Span classes defined.')

Trace and Span classes defined.


## Section 2: Structured Logging

### What is Structured Logging?

Structured logging emits log records as machine-parseable JSON objects rather than
free-form text strings. Every field is a named key-value pair, enabling filtering,
aggregation, and alerting without regex parsing.

**Plain text log (unstructured):**
```
2026-07-01 09:12:03 ERROR Retrieval failed for query milk price after 1450ms
```
This is hard to query. To find all retrieval errors in a time window requires string parsing.

**Structured log (JSON):**
```json
{
  "ts": "2026-07-01T09:12:03Z",
  "level": "ERROR",
  "trace_id": "a3f1c...",
  "span": "retrieval",
  "event": "retrieval_error",
  "duration_ms": 1450,
  "query": "milk price",
  "error": "Pinecone timeout"
}
```
Every field is queryable with `WHERE span='retrieval' AND level='ERROR'`.

**Required fields for every structured log line:**

| Field | Value | Why |
|---|---|---|
| ts | ISO8601 UTC | Enables time-series queries |
| level | DEBUG/INFO/WARNING/ERROR | Severity filtering |
| trace_id | UUID | Correlates all events for one request |
| span | span name | Groups events by pipeline stage |
| event | event name | Identifies what happened |

**What to remember:**
- Log at INFO level for every span start/end. Log at WARNING for slow spans.
  Log at ERROR for exceptions.
- Include trace_id in every exception traceback. Without it you cannot link
  an exception to the user query that caused it.
- Do not log PII (user names, addresses, full queries in production logs).
  Hash or truncate sensitive fields.
- Set log level per environment: DEBUG in dev, INFO in staging, WARNING in prod.

In [21]:
import logging

class StructuredLogger:
    """JSON structured logger with trace context."""

    def __init__(self, name: str, level: str = 'INFO'):
        self._logger = logging.getLogger(name)
        self._logger.setLevel(getattr(logging, level))
        if not self._logger.handlers:
            h = logging.StreamHandler()
            h.setFormatter(logging.Formatter('%(message)s'))
            self._logger.addHandler(h)

    def _emit(self, level: str, trace_id: str, span: str, event: str, **kwargs):
        record = {
            'ts':       datetime.now(timezone.utc).isoformat(),
            'level':    level,
            'trace_id': trace_id,
            'span':     span,
            'event':    event,
            **kwargs,
        }
        getattr(self._logger, level.lower())(json.dumps(record))

    def info(self, trace_id, span, event, **kw):    self._emit('INFO',    trace_id, span, event, **kw)
    def warning(self, trace_id, span, event, **kw): self._emit('WARNING', trace_id, span, event, **kw)
    def error(self, trace_id, span, event, **kw):   self._emit('ERROR',   trace_id, span, event, **kw)

log = StructuredLogger('walmart_agent')

# Demo
demo_trace_id = str(uuid.uuid4())[:8]
log.info(demo_trace_id, 'retrieval', 'span_start',  query='milk price', k=3)
log.info(demo_trace_id, 'retrieval', 'span_end',    duration_ms=210, chunks_returned=3)
log.warning(demo_trace_id, 'llm_call', 'slow_response', duration_ms=3200, threshold_ms=2000)
log.error(demo_trace_id, 'tool_dispatch', 'tool_timeout', tool='price_lookup', timeout_ms=5000)

{"ts": "2026-07-24T05:19:03.113235+00:00", "level": "INFO", "trace_id": "1a4ad0ee", "span": "retrieval", "event": "span_start", "query": "milk price", "k": 3}
{"ts": "2026-07-24T05:19:03.113699+00:00", "level": "INFO", "trace_id": "1a4ad0ee", "span": "retrieval", "event": "span_end", "duration_ms": 210, "chunks_returned": 3}
{"ts": "2026-07-24T05:19:03.113902+00:00", "level": "WARNING", "trace_id": "1a4ad0ee", "span": "llm_call", "event": "slow_response", "duration_ms": 3200, "threshold_ms": 2000}
{"ts": "2026-07-24T05:19:03.114143+00:00", "level": "ERROR", "trace_id": "1a4ad0ee", "span": "tool_dispatch", "event": "tool_timeout", "tool": "price_lookup", "timeout_ms": 5000}


## Section 3: Latency Attribution

### What is Latency Attribution?

Latency attribution is the practice of breaking down total end-to-end latency
into contributions from each span so you can identify the bottleneck.

**Why it matters:**
If a request takes 4 seconds, you need to know whether the latency is in
the retrieval step (network + vector DB), the LLM (model TTFT + generation),
or the tool dispatch (external API). Each bottleneck has a different fix.

**Latency budget framework (Walmart Retail Assistant SLO: P95 < 3s):**

| Stage | Budget | Alert Threshold |
|---|---|---|
| Retrieval (Pinecone) | 300 ms | > 500 ms |
| LLM call (gpt-4-turbo) | 1500 ms | > 2500 ms |
| Tool dispatch | 500 ms | > 1000 ms |
| Response assembly | 100 ms | > 200 ms |
| Total | 2400 ms | > 3000 ms (SLO breach) |

**Latency percentiles:**

| Percentile | Use |
|---|---|
| P50 (median) | Typical user experience |
| P95 | The SLO threshold -- 95% of requests must be under this |
| P99 | Worst-case outliers that damage user trust |

**What to remember:**
- Always measure latency at the span level, not just end-to-end.
- Report P95, not average. Averages hide tail latency.
- A P99 that is 10x the P50 indicates an intermittent external dependency issue.

In [22]:
import statistics

LATENCY_BUDGETS = {
    'retrieval':    300,
    'llm_call':    1500,
    'tool_dispatch': 500,
    'response':     100,
}
LATENCY_ALERTS = {
    'retrieval':    500,
    'llm_call':    2500,
    'tool_dispatch':1000,
    'response':     200,
}
SLO_P95_MS = 3000

def latency_report(traces: list) -> dict:
    span_latencies = {}
    total_latencies = []

    for t in traces:
        total_latencies.append(t.get('total_duration_ms', 0))
        for span in t.get('spans', []):
            name = span['name']
            span_latencies.setdefault(name, [])
            span_latencies[name].append(span['duration_ms'])

    report = {'slo_target_p95_ms': SLO_P95_MS}
    if total_latencies:
        s = sorted(total_latencies)
        n = len(s)
        p95_idx = int(n * 0.95)
        report['end_to_end'] = {
            'p50_ms': round(statistics.median(s), 1),
            'p95_ms': s[min(p95_idx, n-1)],
            'p99_ms': s[min(int(n * 0.99), n-1)],
            'avg_ms': round(statistics.mean(s), 1),
            'slo_breach': s[min(p95_idx, n-1)] > SLO_P95_MS,
        }

    report['spans'] = {}
    for span_name, lats in span_latencies.items():
        budget  = LATENCY_BUDGETS.get(span_name, 999999)
        alert   = LATENCY_ALERTS.get(span_name, 999999)
        avg_lat = round(statistics.mean(lats), 1)
        report['spans'][span_name] = {
            'avg_ms':        avg_lat,
            'budget_ms':     budget,
            'over_budget':   avg_lat > budget,
            'alert_flag':    avg_lat > alert,
            'pct_of_budget': round(avg_lat / budget * 100, 1),
        }

    return report

# Demo: simulate 5 traces with span data
import random
random.seed(42)

def fake_trace(session_idx: int) -> dict:
    spans = [
        {'name': 'retrieval',     'duration_ms': random.randint(180, 420), 'status': 'success'},
        {'name': 'llm_call',      'duration_ms': random.randint(900, 2800), 'status': 'success'},
        {'name': 'tool_dispatch', 'duration_ms': random.randint(300, 950), 'status': 'success'},
        {'name': 'response',      'duration_ms': random.randint(50, 180), 'status': 'success'},
    ]
    total = sum(s['duration_ms'] for s in spans)
    return {'session_id': f'S-{session_idx}', 'total_duration_ms': total, 'spans': spans}

fake_traces = [fake_trace(i) for i in range(20)]
lat_rpt = latency_report(fake_traces)

print('Latency Attribution Report:')
print(f'  End-to-end P95: {lat_rpt["end_to_end"]["p95_ms"]} ms (SLO: {SLO_P95_MS} ms)')
print(f'  SLO breach: {lat_rpt["end_to_end"]["slo_breach"]}')
print()
print(f'  {"Stage":<16} {"Avg (ms)":<12} {"Budget":<10} {"% Budget":<12} Alert')
print('  ' + '-' * 55)
for name, d in lat_rpt['spans'].items():
    flag = 'FLAG' if d['alert_flag'] else ''
    print(f'  {name:<16} {d["avg_ms"]:<12} {d["budget_ms"]:<10} {d["pct_of_budget"]:<12} {flag}')

Latency Attribution Report:
  End-to-end P95: 3699 ms (SLO: 3000 ms)
  SLO breach: True

  Stage            Avg (ms)     Budget     % Budget     Alert
  -------------------------------------------------------
  retrieval        324.4        300        108.1        
  llm_call         1639.8       1500       109.3        
  tool_dispatch    622.4        500        124.5        
  response         109.8        100        109.8        


## Section 4: Token Cost per Trace

### What is Token Cost Tracking?

Every LLM API call consumes tokens and incurs a monetary cost.
In production, a Walmart assistant handling 100,000 queries per day can easily
spend thousands of dollars on token costs. Tracking cost per trace lets you:

- Identify expensive query categories
- Detect prompt bloat (unexplained token growth over time)
- Set per-user or per-session cost budgets
- Make informed model swap decisions (gpt-4-turbo vs gpt-4o-mini cost ratio: ~10x)

**OpenAI pricing reference (as of mid-2026, verify current rates):**

| Model | Input (per 1M tokens) | Output (per 1M tokens) |
|---|---|---|
| gpt-4-turbo | $10.00 | $30.00 |
| gpt-4o | $5.00 | $15.00 |
| gpt-4o-mini | $0.15 | $0.60 |

**What to remember:**
- Track input and output tokens separately. Output tokens cost 3x more on most models.
- System prompt tokens are charged on every request. A 2,000-token system prompt
  at 100,000 daily requests = 200 million daily input tokens = $2,000/day on gpt-4-turbo.
- Context window efficiency: more context means better answers but exponentially more cost.
  RAG chunks must be trimmed to only what is needed.

In [23]:
# Token pricing (USD per 1M tokens) -- update as rates change
MODEL_PRICING = {
    'gpt-4-turbo': {'input': 10.00, 'output': 30.00},
    'gpt-4o':      {'input':  5.00, 'output': 15.00},
    'gpt-4o-mini': {'input':  0.15, 'output':  0.60},
}

def compute_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Return cost in USD for one LLM call."""
    if model not in MODEL_PRICING:
        return 0.0
    p = MODEL_PRICING[model]
    return round(
        (input_tokens  / 1_000_000) * p['input'] +
        (output_tokens / 1_000_000) * p['output'],
        6,
    )

# Simulate a traced LLM call
def traced_llm_call(query: str, context: str, model: str, trace: 'Trace') -> dict:
    span = trace.new_span('llm_call', {'model': model})
    span.add_event('request_sent', {'query_len': len(query)})
    system_prompt = (
        'You are the Walmart Retail Assistant. Answer using only the provided context. '
        'Be concise and accurate.'
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': f'Context: {context}\n\nQuestion: {query}'},
        ],
        temperature=0,
        max_tokens=150,
        timeout=10,
    )
    in_tok  = resp.usage.prompt_tokens
    out_tok = resp.usage.completion_tokens
    cost    = compute_cost(model, in_tok, out_tok)
    answer  = resp.choices[0].message.content
    span.add_event('response_received', {'input_tokens': in_tok, 'output_tokens': out_tok})
    span.finish('success', {
        'input_tokens':  in_tok,
        'output_tokens': out_tok,
        'cost_usd':      cost,
        'answer_len':    len(answer),
    })
    log.info(trace.trace_id, 'llm_call', 'completed',
             model=model, in_tok=in_tok, out_tok=out_tok, cost_usd=cost,
             duration_ms=span.duration_ms)
    return {'answer': answer, 'input_tokens': in_tok, 'output_tokens': out_tok, 'cost_usd': cost}

# Run a traced request
session_trace = Trace(session_id='S-DEMO-01', user_query='What is the price of Great Value Milk?')

# Span 1: retrieval (simulated)
r_span = session_trace.new_span('retrieval', {'k': 3})
time.sleep(0.05)
r_span.finish('success', {'chunks_returned': 3, 'top_score': 0.89})

# Span 2: LLM call (live)
ctx = 'Great Value Whole Milk 1 gallon is priced at $3.98, located in Aisle 12.'
llm_result = traced_llm_call(session_trace.user_query, ctx, 'gpt-4-turbo', session_trace)

# Span 3: response
resp_span = session_trace.new_span('response')
time.sleep(0.01)
resp_span.finish('success', {'answer_chars': len(llm_result['answer'])})

session_trace.finish('success')

t = session_trace.to_dict()
print(f'Trace {t["trace_id"][:8]}...')
print(f'  Query:       {t["user_query"]}')
print(f'  Total time:  {t["total_duration_ms"]} ms')
print(f'  Spans:       {t["span_count"]}')
print()
print(f'  Answer: {llm_result["answer"]}')
print(f'  Tokens: {llm_result["input_tokens"]} in + {llm_result["output_tokens"]} out')
print(f'  Cost:   ${llm_result["cost_usd"]:.6f}')

# Daily cost projection
daily_queries = 100_000
daily_cost    = round(llm_result['cost_usd'] * daily_queries, 2)
print(f'  Projected daily cost at {daily_queries:,} queries: ${daily_cost:.2f}')

{"ts": "2026-07-24T05:19:06.224483+00:00", "level": "INFO", "trace_id": "b751561d-2132-45de-9ade-e32955ffd855", "span": "llm_call", "event": "completed", "model": "gpt-4-turbo", "in_tok": 65, "out_tok": 16, "cost_usd": 0.00113, "duration_ms": 3024.59}


Trace b751561d...
  Query:       What is the price of Great Value Milk?
  Total time:  3092.55 ms
  Spans:       3

  Answer: The price of Great Value Whole Milk 1 gallon is $3.98.
  Tokens: 65 in + 16 out
  Cost:   $0.001130
  Projected daily cost at 100,000 queries: $113.00


## Section 5: Langfuse -- Production Observability Platform

### What is Langfuse?

Langfuse is an open-source observability platform designed specifically for LLM applications.
It provides a dashboard for traces, spans, token costs, evaluation scores,
prompt versioning, and user session analytics.

**Key capabilities:**

| Capability | What it provides |
|---|---|
| Trace explorer | View any trace with full span breakdown and metadata |
| Score logging | Attach evaluation scores (faithfulness, relevancy) to traces |
| Cost analytics | Cumulative token spend by model, user, or session |
| Prompt management | Version-controlled prompts with A/B comparison |
| Dataset management | Store golden dataset records and benchmark runs |
| Alerting | Webhook triggers on score drops or latency spikes |

**Integration pattern:**
```
Langfuse SDK  -->  Langfuse Server (cloud or self-hosted)  -->  Dashboard
```
The SDK sends traces asynchronously via background threads.
Production performance is not affected even if Langfuse is temporarily unavailable.

**Self-hosting vs Cloud:**

| Factor | Langfuse Cloud | Self-hosted |
|---|---|---|
| Setup | Zero | Docker Compose or Kubernetes |
| Data residency | Vendor infrastructure | Full control |
| Cost | Free tier, then usage-based | Infrastructure cost only |
| Enterprise requirement | PII / compliance concerns push toward self-hosted | |

**Required environment variables:** `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`

**What to remember:**
- Always implement a fallback logger. If Langfuse is unavailable at import time
  or the keys are missing, the application must not crash.
- Flush traces before shutdown. The SDK batches traces; unflushed batches
  are lost if the process exits without calling `langfuse.flush()`.
- Tag every trace with environment: dev / staging / prod.
  Mixing environments in one Langfuse project pollutes your dashboards.

In [24]:
# Langfuse integration with graceful fallback

LANGFUSE_AVAILABLE = False
langfuse_client    = None

if LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    try:
        from langfuse import Langfuse
        langfuse_client = Langfuse(
            public_key=LANGFUSE_PUBLIC_KEY,
            secret_key=LANGFUSE_SECRET_KEY,
        )
        LANGFUSE_AVAILABLE = True
        print('Langfuse: connected')
    except Exception as exc:
        print(f'Langfuse: import error ({exc}) -- using fallback logger')
else:
    print('Langfuse: keys not set -- using fallback logger')


class FallbackObservabilityLogger:
    """Writes trace records to a local JSONL file when Langfuse is unavailable."""
    def __init__(self, log_path: str = 'observability_fallback.jsonl'):
        self.path = Path(log_path)

    def log_trace(self, trace: dict):
        with self.path.open('a') as f:
            f.write(json.dumps(trace) + '\n')
        log.info(trace.get('trace_id','?'), 'observability', 'trace_logged',
                 backend='fallback_jsonl', path=str(self.path))

    def log_score(self, trace_id: str, metric: str, value: float):
        record = {'trace_id': trace_id, 'metric': metric, 'value': value,
                  'ts': datetime.now(timezone.utc).isoformat()}
        with self.path.open('a') as f:
            f.write(json.dumps(record) + '\n')

fallback_logger = FallbackObservabilityLogger()

def observe_trace(trace_dict: dict, scores: dict = None):
    """Send trace and optional scores to Langfuse or fallback logger."""
    if LANGFUSE_AVAILABLE and langfuse_client:
        # Langfuse v4 API uses trace_context + observation/event primitives.
        lf_trace_id = str(trace_dict['trace_id']).replace('-', '').lower() # 52379C63-5BD4-48E2 -> 52379c635bd448e2

        langfuse_client.create_event(
            trace_context={'trace_id': lf_trace_id},
            name='walmart_retail_assistant',
            input={'query': trace_dict.get('user_query')},
            metadata={k: v for k, v in trace_dict.items() if k != 'spans'},
        )

        for span in trace_dict.get('spans', []):
            lf_span = langfuse_client.start_observation(
                trace_context={'trace_id': lf_trace_id},
                name=span['name'],
                as_type='span',
                metadata=span.get('metadata', {}),
            )
            lf_span.end()

    #     {
    #       'faithfulness': 0.93,
    #       'answer_relevancy': 0.88
    #     }
        if scores:
            for metric, value in scores.items():
                langfuse_client.create_score(
                    trace_id=lf_trace_id,
                    name=metric,
                    value=value,
                )
    else:
        fallback_logger.log_trace(trace_dict)
        if scores:
            for metric, value in scores.items():
                fallback_logger.log_score(trace_dict['trace_id'], metric, value)

# Log the session trace from the previous cell
observe_trace(session_trace.to_dict(), scores={'faithfulness': 0.93, 'answer_relevancy': 0.88})
print('Trace observation complete.')
print(f'Backend used: {"Langfuse" if LANGFUSE_AVAILABLE else "Fallback JSONL"}')

Langfuse: connected
Trace observation complete.
Backend used: Langfuse


## Section 6: Root-Cause Debugging with Traces

### The Debugging Workflow for Agentic Systems

When a user reports a bad answer, a standard debugging workflow using traces is:

```
Step 1: Retrieve the trace_id from the user complaint or support ticket
Step 2: Pull all spans for that trace from Langfuse or the fallback JSONL
Step 3: Check retrieval span -- were the right chunks returned?
Step 4: Check the LLM input -- was the context clean? Was the system prompt correct?
Step 5: Check the LLM output -- was the raw answer correct before post-processing?
Step 6: Check tool dispatch -- if a tool was called, did it return the expected value?
Step 7: Check evaluation scores -- did faithfulness or relevancy flag this trace?
```

**Common failure patterns in Walmart Retail Assistant:**

| Symptom | Likely cause | Debug signal |
|---|---|---|
| Wrong price in answer | Retrieval returned wrong product chunk | Check top chunk content in retrieval span |
| Answer says 'I don't know' | No relevant chunk retrieved (hit rate = 0) | Check chunks_returned = 0 in retrieval span |
| Answer is truncated | max_tokens too low | Check output_tokens == max_tokens in llm_call span |
| Very high latency | LLM timeout or retry | Check duration_ms in llm_call span |
| Tool called with wrong argument | Prompt template error | Check tool_dispatch span input in metadata |

In [25]:
def debug_trace(trace_dict: dict) -> dict:
    """Analyse a trace dict for common failure patterns."""
    issues  = []
    signals = {}

    for span in trace_dict.get('spans', []):
        m = span.get('metadata', {})
        name = span['name']

        if name == 'retrieval':
            chunks = m.get('chunks_returned', 1)
            score  = m.get('top_score', 1.0)
            signals['retrieval_chunks'] = chunks
            signals['retrieval_top_score'] = score
            if chunks == 0:
                issues.append('RETRIEVAL_MISS: zero chunks returned -- query may be out-of-vocabulary')
            if score < 0.6:
                issues.append(f'LOW_RETRIEVAL_SCORE: top score {score} < 0.6 -- context may be irrelevant')
            if span['duration_ms'] > LATENCY_ALERTS.get('retrieval', 500):
                issues.append(f'RETRIEVAL_SLOW: {span["duration_ms"]}ms exceeds alert threshold')

        if name == 'llm_call':
            in_tok  = m.get('input_tokens', 0)
            out_tok = m.get('output_tokens', 0)
            signals['input_tokens']  = in_tok
            signals['output_tokens'] = out_tok
            if span['duration_ms'] > LATENCY_ALERTS.get('llm_call', 2500):
                issues.append(f'LLM_SLOW: {span["duration_ms"]}ms exceeds alert threshold')
            if in_tok > 3000:
                issues.append(f'TOKEN_BLOAT: {in_tok} input tokens -- check context size')

        if span['status'] == 'error':
            issues.append(f'SPAN_ERROR: {name} reported error status')

    return {
        'trace_id': trace_dict.get('trace_id', '?')[:8],
        'issues':   issues if issues else ['No issues detected'],
        'signals':  signals,
    }

# Debug the session trace
debug_result = debug_trace(session_trace.to_dict())
print(f'Debug report for trace {debug_result["trace_id"]}...')
print(f'  Issues:  {debug_result["issues"]}')
print(f'  Signals: {debug_result["signals"]}')

# Simulate a problem trace (retrieval miss)
problem_trace = Trace('S-PROBLEM', 'What is the price of Wagyu beef?')
bad_r_span = problem_trace.new_span('retrieval', {'k': 3})
time.sleep(0.01)
bad_r_span.finish('success', {'chunks_returned': 0, 'top_score': 0.28})
bad_llm_span = problem_trace.new_span('llm_call', {'model': 'gpt-4-turbo'})
time.sleep(0.01)
bad_llm_span.finish('success', {'input_tokens': 450, 'output_tokens': 32})
problem_trace.finish('success')

problem_debug = debug_trace(problem_trace.to_dict())
print()
print(f'Debug report for problem trace {problem_debug["trace_id"]}...')
for issue in problem_debug['issues']:
    print(f'  [ISSUE] {issue}')

Debug report for trace b751561d...
  Issues:  ['LLM_SLOW: 3024.59ms exceeds alert threshold']
  Signals: {'retrieval_chunks': 3, 'retrieval_top_score': 0.89, 'input_tokens': 65, 'output_tokens': 16}

Debug report for problem trace edfe2d7e...
  [ISSUE] RETRIEVAL_MISS: zero chunks returned -- query may be out-of-vocabulary
  [ISSUE] LOW_RETRIEVAL_SCORE: top score 0.28 < 0.6 -- context may be irrelevant


## Section 7: Evaluation Scorecard

The evaluation scorecard aggregates all metric scores from IN10 and IN11
into a single reference document suitable for stakeholder review.
It serves as the acceptance gate before a model version can be promoted to production.

In [26]:
# Load scores from IN10 and IN11
IN10_PATH = Path('in10_eval_scores.json')
IN11_PATH = Path('in11_benchmark_report.json')

in10_scores = json.loads(IN10_PATH.read_text()) if IN10_PATH.exists() else {}
in11_report = json.loads(IN11_PATH.read_text()) if IN11_PATH.exists() else {}

if in10_scores:
    print(f'IN10 scores loaded: {list(in10_scores.keys())}')
else:
    print('IN10 scores not found -- run IN10 first, or using placeholder values')
    in10_scores = {
        'faithfulness':       {'score': 0.88, 'threshold': 0.85, 'pass': True},
        'answer_relevancy':   {'score': 0.80, 'threshold': 0.75, 'pass': True},
        'toxicity':           {'score': 0.00, 'threshold': 0.10, 'pass': True},
        'hit_rate_at_3':      {'score': 0.85, 'threshold': 0.75, 'pass': True},
        'mrr':                {'score': 0.73, 'threshold': 0.65, 'pass': True},
        'precision_at_3':     {'score': 0.62, 'threshold': 0.55, 'pass': True},
        'context_precision':  {'score': 0.71, 'threshold': 0.65, 'pass': True},
        'task_success_rate':  {'score': 0.93, 'threshold': 0.90, 'pass': True},
        'tool_call_accuracy': {'score': 0.97, 'threshold': 0.95, 'pass': True},
        'step_completion_ratio': {'score': 0.94, 'threshold': 0.92, 'pass': True},
    }

if in11_report:
    print(f'IN11 report loaded: decision={in11_report.get("decision")}')
else:
    print('IN11 report not found -- using placeholder')
    in11_report = {
        'baseline_model':        'gpt-4-turbo',
        'comparison_model':      'gpt-4o-mini',
        'baseline_avg_score':    0.87,
        'comparison_avg_score':  0.83,
        'aggregate_delta':      -0.04,
        'latency_delta_ms':     -320,
        'decision':             'CONDITIONAL -- minor regression, review in staging before deploy',
    }

passed  = sum(1 for v in in10_scores.values() if v.get('pass', False))
total   = len(in10_scores)
overall = 'PASS' if passed == total else 'FAIL'
print(f'Overall: {passed}/{total} metrics passed -- {overall}')

IN10 scores loaded: ['faithfulness', 'answer_relevancy', 'toxicity', 'hit_rate_at_3', 'mrr_at_3', 'precision_at_3', 'context_precision', 'task_success_rate', 'tool_call_accuracy', 'step_completion_ratio']
IN11 report loaded: decision=BLOCKED -- critical category regression
Overall: 9/10 metrics passed -- FAIL


In [27]:
# Generate evaluation scorecard text file
lines = [
    'WALMART RETAIL ASSISTANT -- EVALUATION SCORECARD',
    '=' * 60,
    f'Generated : {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}',
    f'Programme : Advanced Agentic AI -- Production Engineering',
    f'Module    : 4 -- Evaluation, Observability and Debugging',
    '',
    'SECTION 1: OUTPUT-LEVEL METRICS (IN10)',
    '-' * 60,
]

output_metrics = ['faithfulness', 'answer_relevancy', 'toxicity']
for m in output_metrics:
    d = in10_scores.get(m, {})
    status = 'PASS' if d.get('pass') else 'FAIL'
    lines.append(f'  {m:<28} Score: {d.get("score", "N/A"):<8} Threshold: {d.get("threshold", "N/A"):<8} {status}')

lines += [
    '',
    'SECTION 2: RETRIEVAL-LEVEL METRICS (IN10)',
    '-' * 60,
]
retrieval_metrics = ['hit_rate_at_3', 'mrr', 'precision_at_3', 'context_precision']
for m in retrieval_metrics:
    d = in10_scores.get(m, {})
    status = 'PASS' if d.get('pass') else 'FAIL'
    lines.append(f'  {m:<28} Score: {d.get("score", "N/A"):<8} Threshold: {d.get("threshold", "N/A"):<8} {status}')

lines += [
    '',
    'SECTION 3: AGENT-LEVEL METRICS (IN10)',
    '-' * 60,
]
agent_metrics = ['task_success_rate', 'tool_call_accuracy', 'step_completion_ratio']
for m in agent_metrics:
    d = in10_scores.get(m, {})
    status = 'PASS' if d.get('pass') else 'FAIL'
    lines.append(f'  {m:<28} Score: {d.get("score", "N/A"):<8} Threshold: {d.get("threshold", "N/A"):<8} {status}')

lines += [
    '',
    'SECTION 4: REGRESSION TEST SUMMARY (IN11)',
    '-' * 60,
    f'  Baseline model       : {in11_report.get("baseline_model")}',
    f'  Comparison model     : {in11_report.get("comparison_model")}',
    f'  Baseline avg score   : {in11_report.get("baseline_avg_score")}',
    f'  Comparison avg score : {in11_report.get("comparison_avg_score")}',
    f'  Aggregate delta      : {in11_report.get("aggregate_delta")}',
    f'  Latency delta (ms)   : {in11_report.get("latency_delta_ms")}',
    f'  Decision             : {in11_report.get("decision")}',
    '',
    'SECTION 5: OVERALL GATE',
    '-' * 60,
    f'  Metrics passed : {passed}/{total}',
    f'  Gate status    : {overall}',
    '',
    'RECOMMENDATION',
    '-' * 60,
]

if overall == 'PASS':
    lines.append('  All evaluation gates passed. Proceed per regression test decision.')
else:
    failed = [m for m, d in in10_scores.items() if not d.get('pass', True)]
    lines.append(f'  BLOCKED: {len(failed)} metric(s) below threshold: {failed}')
    lines.append('  Address failing metrics before promotion to production.')

scorecard_text = '\n'.join(lines)
Path('evaluation_scorecard.txt').write_text(scorecard_text)
print(scorecard_text)

WALMART RETAIL ASSISTANT -- EVALUATION SCORECARD
Generated : 2026-07-24 05:19 UTC
Programme : Advanced Agentic AI -- Production Engineering
Module    : 4 -- Evaluation, Observability and Debugging

SECTION 1: OUTPUT-LEVEL METRICS (IN10)
------------------------------------------------------------
  faithfulness                 Score: 1.0      Threshold: 0.85     PASS
  answer_relevancy             Score: 0.9      Threshold: 0.75     PASS
  toxicity                     Score: 0.0      Threshold: 0.1      PASS

SECTION 2: RETRIEVAL-LEVEL METRICS (IN10)
------------------------------------------------------------
  hit_rate_at_3                Score: 0.75     Threshold: 0.75     PASS
  mrr                          Score: N/A      Threshold: N/A      FAIL
  precision_at_3               Score: 0.6      Threshold: 0.55     PASS
  context_precision            Score: 0.858    Threshold: 0.65     PASS

SECTION 3: AGENT-LEVEL METRICS (IN10)
-------------------------------------------------------

In [28]:
# Generate observability dashboard design document
dash_lines = [
    'WALMART RETAIL ASSISTANT -- OBSERVABILITY DASHBOARD DESIGN',
    '=' * 65,
    f'Version   : 1.0',
    f'Date      : {datetime.now(timezone.utc).strftime("%Y-%m-%d")}',
    '',
    'PURPOSE',
    '-' * 65,
    '  This document specifies the panels, metrics, and alert rules',
    '  for the production Langfuse observability dashboard.',
    '',
    'DASHBOARD LAYOUT (3 rows x 4 columns)',
    '-' * 65,
    '',
    'ROW 1: HEALTH OVERVIEW',
    '  Panel 1.1: Real-time request rate (requests/min, last 1 hour)',
    '  Panel 1.2: Error rate % (last 1 hour, red if > 2%)',
    '  Panel 1.3: P95 end-to-end latency (ms, red if > 3000ms)',
    '  Panel 1.4: Cumulative token cost today (USD)',
    '',
    'ROW 2: QUALITY METRICS',
    '  Panel 2.1: Faithfulness score trend (7-day rolling avg)',
    '  Panel 2.2: Answer relevancy score trend (7-day rolling avg)',
    '  Panel 2.3: Task success rate by category (bar chart)',
    '  Panel 2.4: Tool call accuracy (last 24h)',
    '',
    'ROW 3: LATENCY BREAKDOWN',
    '  Panel 3.1: Avg latency per span (retrieval / llm / tool / response)',
    '  Panel 3.2: Latency heatmap by hour-of-day',
    '  Panel 3.3: P99 latency trend (7-day)',
    '  Panel 3.4: SLO compliance % (% requests under 3000ms)',
    '',
    'ALERT RULES',
    '-' * 65,
    '  ALERT-01: Error rate > 2% for 5 consecutive minutes',
    '            Action: Page on-call, auto-rollback if > 5%',
    '',
    '  ALERT-02: P95 latency > 3000ms (SLO breach) for 3 consecutive minutes',
    '            Action: Page on-call, open P1 incident',
    '',
    '  ALERT-03: Faithfulness score drops below 0.80 (rolling 1h avg)',
    '            Action: Page ML team, enable shadow mode with baseline model',
    '',
    '  ALERT-04: Daily token cost exceeds $1,000',
    '            Action: Notify ML team, review prompt length and caching',
    '',
    '  ALERT-05: Retrieval span avg latency > 500ms',
    '            Action: Check Pinecone index health and network egress',
    '',
    'RETENTION POLICY',
    '-' * 65,
    '  Raw trace data: 30 days (GDPR/privacy compliance)',
    '  Aggregated metrics: 1 year',
    '  Evaluation scores: 1 year (linked to model version)',
    '  Benchmark reports: permanent (version-controlled)',
    '',
    'TOOLING',
    '-' * 65,
    '  Primary observability: Langfuse (self-hosted, Docker on K8s)',
    '  Alerting: PagerDuty via Langfuse webhooks',
    '  Log archive: S3 + Athena (JSONL, partitioned by date)',
    '  Cost tracking: custom TokenCostTracker + Grafana',
]

dashboard_text = '\n'.join(dash_lines)
Path('observability_dashboard_design.txt').write_text(dashboard_text)
print(dashboard_text)

WALMART RETAIL ASSISTANT -- OBSERVABILITY DASHBOARD DESIGN
Version   : 1.0
Date      : 2026-07-24

PURPOSE
-----------------------------------------------------------------
  This document specifies the panels, metrics, and alert rules
  for the production Langfuse observability dashboard.

DASHBOARD LAYOUT (3 rows x 4 columns)
-----------------------------------------------------------------

ROW 1: HEALTH OVERVIEW
  Panel 1.1: Real-time request rate (requests/min, last 1 hour)
  Panel 1.2: Error rate % (last 1 hour, red if > 2%)
  Panel 1.3: P95 end-to-end latency (ms, red if > 3000ms)
  Panel 1.4: Cumulative token cost today (USD)

ROW 2: QUALITY METRICS
  Panel 2.1: Faithfulness score trend (7-day rolling avg)
  Panel 2.2: Answer relevancy score trend (7-day rolling avg)
  Panel 2.3: Task success rate by category (bar chart)
  Panel 2.4: Tool call accuracy (last 24h)

ROW 3: LATENCY BREAKDOWN
  Panel 3.1: Avg latency per span (retrieval / llm / tool / response)
  Panel 3.2: Latency

## Summary

| Concept | Key Rule |
|---|---|
| Trace / Span / Event | Hierarchy for agentic system observability; trace_id in every log line |
| Structured logging | JSON records with ts, level, trace_id, span, event fields |
| Latency attribution | Measure P95, not average; budget per stage, alert on overage |
| Token cost | Track input + output separately; output tokens cost 3x more |
| Langfuse | Production observability; always implement graceful fallback |
| Root-cause debugging | Use span metadata to narrow failure to retrieval, LLM, or tool |
| Evaluation scorecard | Aggregates IN10 + IN11 scores into a stakeholder gate document |
| Dashboard design | 3-row layout: health / quality / latency; 5 alert rules |

**Module 4 complete.** Deliverables generated:
- `evaluation_scorecard.txt` -- pass/fail gate across all 10 evaluation metrics
- `observability_dashboard_design.txt` -- production dashboard specification
- `in10_eval_scores.json` -- metric scores from IN10
- `in11_benchmark_report.json` -- regression test report from IN11
- `observability_fallback.jsonl` -- local trace log (if Langfuse not configured)